In [3]:
import pandas as pd

df = pd.read_parquet('./data/yellow_tripdata_2023-01.parquet')
df.shape


(3066766, 19)

In [4]:
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

df.duration.std()


np.float64(42.59435124195457)

In [5]:
total_antes = len(df)
df = df[(df.duration >= 1) & (df.duration <= 60)]
total_despues = len(df)

total_despues / total_antes


0.9812202822125979

In [6]:
from sklearn.feature_extraction import DictVectorizer

categorical = ['PULocationID', 'DOLocationID']
df[categorical] = df[categorical].astype(str)

train_dicts = df[categorical].to_dict(orient='records')
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

X_train.shape


(3009173, 515)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

target = 'duration'
y_train = df[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)
root_mean_squared_error(y_train, y_pred)


7.64926193410655

In [8]:
df_val = pd.read_parquet('./data/yellow_tripdata_2023-02.parquet')

df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]

df_val[categorical] = df_val[categorical].astype(str)

val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_val = df_val[target].values
y_pred_val = lr.predict(X_val)
root_mean_squared_error(y_val, y_pred_val)


7.811817487693915